In [ ]:
"""
====================================================================================
Sandwich Attack Triplet Detection in DeFi Transactions
====================================================================================

Purpose
-------
This function identifies potential sandwich attacks in Ethereum-based DeFi
transactions. Sandwich attacks involve a sequence of three trades (front-run,
victim, back-run) executed by the same attacker to profit from the victim's
trade.

Logic
-----
1. Normalize relevant token and address fields to lowercase for robust comparison.
2. Sort transactions by block number and transaction order within the block.
3. Build a mapping from block numbers to row indices for efficient block-level search.
4. Iterate through each transaction as a potential front-run (t1):
   - Ensure t1 has required fields and token information.
   - Candidate victim transactions (t2) are searched within the same block:
       * Must involve the same token pair as t1.
       * Must not be initiated by the attacker.
       * Must occur after t1 in the transaction order.
   - Candidate back-run transactions (t3) are searched within the same block:
       * Must originate from the same attacker.
       * Must reverse the token swap executed in t1.
5. Store all detected sandwich triplets with relevant metadata:
   - Transaction indices and hashes
   - Tokens and their corresponding values
   - Block number and timestamps
   - Attacker and victim addresses
   - Flags for Sybil or miner involvement
6. Return a DataFrame with all detected sandwich triplets.

Inputs
------
- `df`: A DataFrame containing DeFi transaction data with at least the following
  columns:
    * 'fromaddress', 'token_in', 'token_out', 'value_token_in', 'value_token_out',
      'transaction_hash', 'block_number', 'tx_order_in_block', 'detecttime',
      optionally 'is_sybil' and 'is_miner'.

Outputs
-------
- A DataFrame containing all detected sandwich attack triplets with detailed
  metadata for analysis and further processing.
"""


In [ ]:
import pandas as pd
from tqdm import tqdm

def detect_sandwich_triplets(df):
    df = df.copy()
    df['detecttime'] = pd.to_datetime(df['detecttime'], errors='coerce')

    for col in ['token_in', 'token_out']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: x.lower() if isinstance(x, str) else x)
        else:
            raise ValueError(f"La colonne requise '{col}' est absente du DataFrame.")

    df['fromaddress'] = df['fromaddress'].apply(lambda x: x.lower() if isinstance(x, str) else x)
    df = df.sort_values(['block_number', 'tx_order_in_block']).reset_index(drop=True)
    n = len(df)

    block_to_indices = {}
    for i, block in enumerate(df['block_number']):
        block_to_indices.setdefault(int(block), []).append(i)

    results = []
    for i in tqdm(range(n), desc="Détection sandwich améliorée"):
        t1 = df.iloc[i]
        made_by_sybil = 0 if pd.isna(t1.get('is_sybil')) else int(t1['is_sybil'])
        made_by_miner = 0 if pd.isna(t1.get('is_miner')) else int(t1['is_miner'])

        attacker = t1['fromaddress']
        token_out_t1 = t1['token_out']
        token_in_t1 = t1['token_in']
        block_t1 = int(t1['block_number'])
        time_t1 = t1['detecttime']
        index_t1 = t1['tx_order_in_block']
        value_in_t1 = t1['value_token_in']
        value_out_t1 = t1['value_token_out']

        if pd.isna(time_t1) or token_out_t1 is None:
            continue

        candidate_indices = block_to_indices.get(block_t1, [])
        for j in candidate_indices:
            if j <= i:
                continue
            t2 = df.iloc[j]
            time_t2 = t2['detecttime']
            index_t2 = t2['tx_order_in_block']
            value_in_t2 = t2['value_token_in']
            value_out_t2 = t2['value_token_out']
            token_in_t2 = t2['token_in']
            token_out_t2 = t2['token_out']

            if pd.isna(index_t2) or index_t2 <= index_t1:
                continue
            if token_out_t2 != token_out_t1 or token_in_t2 != token_in_t1:
                continue
            if t2['fromaddress'] == attacker:
                continue

            for k in candidate_indices:
                if k <= j:
                    continue
                t3 = df.iloc[k]
                index_t3 = t3['tx_order_in_block']
                token_in_t3 = t3['token_in']
                token_out_t3 = t3['token_out']
                value_in_t3 = t3['value_token_in']
                value_out_t3 = t3['value_token_out']

                if pd.isna(index_t3) or index_t3 <= index_t2:
                    continue
                if t3['fromaddress'] != attacker:
                    continue
                if token_in_t3 != token_out_t1 or token_out_t3 != token_in_t1:
                    continue

                results.append({
                    'front_idx': index_t1,
                    'front_tx_hash': t1.get('transaction_hash'),
                    't1_token_in': token_in_t1,
                    'front_value_in': value_in_t1,
                    't1_token_out': token_out_t1,
                    'front_value_out': value_out_t1,
                    'victim_idx': index_t2,
                    'victim_tx_hash': t2.get('transaction_hash'),
                    't2_token_in': token_in_t2,
                    'victim_value_in': value_in_t2,
                    't2_token_out': token_out_t2,
                    'victim_value_out': value_out_t2,
                    'back_idx': index_t3,
                    'back_tx_hash': t3.get('transaction_hash'),
                    't3_token_in': token_in_t3,
                    'back_value_in': value_in_t3,
                    't3_token_out': token_out_t3,
                    'back_value_out': value_out_t3,
                    'attacker': attacker,
                    'victim': t2.get('fromaddress'),
                    'token_pivot': token_out_t1,
                    'block': block_t1,
                    'front_time': time_t1,
                    'victim_time': time_t2,
                    'back_time': t3['detecttime'],
                    'made_by_sybil': made_by_sybil,
                    'made_by_miner': made_by_miner
                })

    return pd.DataFrame(results)

sandwiches=detect_sandwich_triplets(cleaned_decoded_merged_transactions_df)

In [ ]:
"""
====================================================================================
Sandwich Attack Profit Calculation
====================================================================================

Purpose
-------
This function calculates the potential profit for the attacker in detected
sandwich attacks based on front-run (t1) and back-run (t3) transactions. It
enriches the sandwich triplet DataFrame with the profit amount, profit token, and
matching method used.

Logic
-----
1. Define a helper function to compare token values with a small tolerance ratio
   (default 1e-6) to account for minor rounding differences in token transfers.
2. Iterate over each detected sandwich triplet:
   - Extract values of front-run input/output and back-run input/output.
   - Attempt exact token match checks:
       * "exchanged": front_out ≈ back_in
       * "pivot": front_in ≈ back_out
   - If exact matches fail, compute profit using a ratio-based fallback method:
       * Estimate expected back-run output based on front-run ratio.
       * Calculate profit as the difference between actual and expected values.
3. Normalize profit by token decimals (default 18) to obtain human-readable amounts.
4. Append the following columns to the DataFrame:
   - `token_match`: indicates the type of token matching used ("exchanged",
     "pivot", "ratio_based", or None if failed)
   - `attacker_profit`: computed profit in token units
   - `profit_token`: the token in which profit is measured

Inputs
------
- `df`: A DataFrame of detected sandwich triplets.
- `default_decimals`: Default number of decimals for tokens (default 18).

Outputs
-------
- A DataFrame with the added columns: `token_match`, `attacker_profit`, `profit_token`.
"""


In [ ]:
import pandas as pd

def tokens_almost_equal(val1, val2, tolerance_ratio=1e-6):
    try:
        val1 = int(val1)
        val2 = int(val2)
    except (ValueError, TypeError):
        return False
    if val1 == 0 or val2 == 0:
        return False
    return abs(val1 - val2) / max(val1, val2) < tolerance_ratio

def add_token_match_and_profit_with_ratio(df: pd.DataFrame, default_decimals=18) -> pd.DataFrame:
    token_match_list = []
    attacker_profit_list = []
    profit_token_list = []

    for _, row in df.iterrows():
        try:
            t1_in = int(row.get('front_value_in'))
            t1_out = int(row.get('front_value_out'))
            t3_in = int(row.get('back_value_in'))
            t3_out = int(row.get('back_value_out'))
        except (TypeError, ValueError):
            token_match_list.append(None)
            attacker_profit_list.append(None)
            profit_token_list.append(None)
            continue

        decimals = row.get('t1_token_in_decimals', default_decimals)
        pivot_token = row.get('token_pivot')
        profit_token = row.get('t1_token_in')

        if tokens_almost_equal(t1_out, t3_in):
            try:
                profit = (t3_out - t1_in) / 10**decimals
                token_match_list.append("exchanged")
                attacker_profit_list.append(profit)
                profit_token_list.append(profit_token)
            except Exception:
                token_match_list.append(None)
                attacker_profit_list.append(None)
                profit_token_list.append(None)

        elif tokens_almost_equal(t1_in, t3_out):
            try:
                profit = (t1_out - t3_in) / 10**decimals
                token_match_list.append("pivot")
                attacker_profit_list.append(profit)
                profit_token_list.append(pivot_token)
            except Exception:
                token_match_list.append(None)
                attacker_profit_list.append(None)
                profit_token_list.append(None)

        else:
            try:
                ratio_t1 = t1_in / t1_out if t1_out != 0 else None
                if ratio_t1 is not None:
                    expected_t3_out = t3_in * ratio_t1
                    profit = (t3_out - expected_t3_out) / 10**decimals
                    token_match_list.append("ratio_based")
                    attacker_profit_list.append(profit)
                    profit_token_list.append(profit_token)
                else:
                    token_match_list.append(None)
                    attacker_profit_list.append(None)
                    profit_token_list.append(None)
            except Exception:
                token_match_list.append(None)
                attacker_profit_list.append(None)
                profit_token_list.append(None)

    df = df.copy()
    df['token_match'] = token_match_list
    df['attacker_profit'] = attacker_profit_list
    df['profit_token'] = profit_token_list
    return df
